In [1]:
import joblib
import pandas as pd
from sklearn import set_config
from tempfile import TemporaryDirectory
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
import warnings

In [2]:
target_column = "health_condition"

In [3]:
X_train = pd.read_csv("../data/intermediate/train_features.csv")
X_valid = pd.read_csv("../data/intermediate/valid_features.csv")
X_test = pd.read_csv("../data/intermediate/test_features.csv")

y_train = pd.read_csv("../data/intermediate/train_labels.csv")
y_valid = pd.read_csv("../data/intermediate/valid_labels.csv")

X_train['diet_type'] = X_train['diet_type'].astype('category')
X_valid['diet_type'] = X_valid['diet_type'].astype('category')
X_test['diet_type'] = X_test['diet_type'].astype('category')

X_train['gender'] = X_train['gender'].astype('category')
X_valid['gender'] = X_valid['gender'].astype('category')
X_test['gender'] = X_test['gender'].astype('category')

X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 552070 entries, 0 to 552069
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype   
---  ------                        --------------   -----   
 0   sleep_duration                491266 non-null  float64 
 1   heart_rate                    545802 non-null  float64 
 2   bmi                           541053 non-null  float64 
 3   calorie_expenditure           509705 non-null  float64 
 4   step_count                    540909 non-null  float64 
 5   exercise_duration             546550 non-null  float64 
 6   water_intake                  517224 non-null  float64 
 7   diet_type                     546580 non-null  category
 8   stress_level                  485727 non-null  float64 
 9   sleep_quality                 505469 non-null  float64 
 10  physical_activity_level       522744 non-null  float64 
 11  smoking_alcohol               529184 non-null  float64 
 12  gender                        535022 non-

In [4]:
X = pd.concat([X_train, X_valid], axis=0)
y = pd.concat([y_train, y_valid], axis=0)

In [5]:
df_submission = pd.read_csv("../data/sample_submission.csv")
df_submission.info()

<class 'pandas.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   id                295753 non-null  int64
 1   health_condition  295753 non-null  str  
dtypes: int64(1), str(1)
memory usage: 4.5 MB


In [6]:
train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
valid_sample_weight = compute_sample_weight(class_weight="balanced", y=y_valid)
y_sample_weight = compute_sample_weight(class_weight="balanced", y=y)

model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=8, num_class=3, objective='multi:softprob', tree_method='hist', enable_categorical=True, early_stopping_rounds=100)
model.fit(X_train, y_train, sample_weight=train_sample_weight, eval_set=[(X_valid, y_valid)])

joblib.dump(model, '../models/xgboost_80.pkl')

[0]	validation_0-mlogloss:0.97723
[1]	validation_0-mlogloss:0.87768
[2]	validation_0-mlogloss:0.79355
[3]	validation_0-mlogloss:0.72169
[4]	validation_0-mlogloss:0.65980
[5]	validation_0-mlogloss:0.60612
[6]	validation_0-mlogloss:0.55936
[7]	validation_0-mlogloss:0.51840
[8]	validation_0-mlogloss:0.48243
[9]	validation_0-mlogloss:0.45079
[10]	validation_0-mlogloss:0.42285
[11]	validation_0-mlogloss:0.39808
[12]	validation_0-mlogloss:0.37613
[13]	validation_0-mlogloss:0.35662
[14]	validation_0-mlogloss:0.33926
[15]	validation_0-mlogloss:0.32379
[16]	validation_0-mlogloss:0.30998
[17]	validation_0-mlogloss:0.29763
[18]	validation_0-mlogloss:0.28654
[19]	validation_0-mlogloss:0.27662
[20]	validation_0-mlogloss:0.26773
[21]	validation_0-mlogloss:0.25974
[22]	validation_0-mlogloss:0.25256
[23]	validation_0-mlogloss:0.24610
[24]	validation_0-mlogloss:0.24029
[25]	validation_0-mlogloss:0.23506
[26]	validation_0-mlogloss:0.23031
[27]	validation_0-mlogloss:0.22604
[28]	validation_0-mlogloss:0.2

['../models/xgboost_80.pkl']

In [ ]:
from sklearn.metrics import balanced_accuracy_score

y_pred = model.predict(X_valid)

val_score = balanced_accuracy_score(y_valid, y_pred, sample_weight=valid_sample_weight)
print("Validation Balanced Accuracy:", val_score)

In [7]:
model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=8, num_class=3, objective='multi:softprob', tree_method='hist', enable_categorical=True)
model.fit(X, y, sample_weight=y_sample_weight)

joblib.dump(model, '../models/xgboost_100.pkl')

['../models/xgboost_100.pkl']

In [8]:
y_pred = model.predict(X_test)

df_submission[target_column] = y_pred
df_submission[target_column] = df_submission[target_column].replace({0:'unhealthy', 1:'at-risk', 2: 'fit'})

df_submission.to_csv('../results/xgboost_baseline.csv', index=False)
df_submission

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
...,...,...
295748,985836,fit
295749,985837,at-risk
295750,985838,unhealthy
295751,985839,at-risk
